In [ ]:
!pip install transformers
!pip install transformers[sentencepiece]

In [ ]:
pip install pysentimiento

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 21.2 MB/s eta 0:00:00


# Tokenizadores de subpalabras

1. Word Level
2. BPE: Byte Pair Encoder
3. Unigram
4. WordPiece

In [ ]:
## importing the tokenizer and subword BPE trainer
from tokenizers import Tokenizer
from tokenizers.models import BPE, Unigram, WordLevel, WordPiece
from tokenizers.trainers import BpeTrainer, WordLevelTrainer, \
                                WordPieceTrainer, UnigramTrainer
from tokenizers.pre_tokenizers import Whitespace

# Dataset wikicorpus desde [Hugging Face](https://huggingface.co)

El dataset es uno de los nuevos compontes en el nuevo modelo de ecosistemas de

In [ ]:
from datasets import load_dataset

dataset = load_dataset("wikicorpus", "raw_es",split='train')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/datasets/load.py:1461: FutureWarning: The repository for wikicorpus contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/wikicorpus
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Generating train split:   0%|          | 0/259409 [00:00<?, ? examples/s]

In [ ]:
print(dataset)
dataset.to_csv('real_news.csv')

Dataset({
    features: ['id', 'title', 'text'],
    num_rows: 259409
})


Creating CSV from Arrow format:   0%|          | 0/260 [00:00<?, ?ba/s]

670286963

In [ ]:
unk_token = "<UNK>"  # token for unknown words
spl_tokens = ["<UNK>", "<SEP>", "<MASK>", "<CLS>"]  # special tokens

def prepare_tokenizer_trainer(alg):
    """
    Prepares the tokenizer and trainer with unknown & special tokens.
    """
    if alg == 'BPE':
        tokenizer = Tokenizer(BPE(unk_token = unk_token))
        trainer = BpeTrainer(special_tokens = spl_tokens)
    elif alg == 'UNI':
        tokenizer = Tokenizer(Unigram())
        trainer = UnigramTrainer(unk_token= unk_token, special_tokens = spl_tokens)
    elif alg == 'WPC':
        tokenizer = Tokenizer(WordPiece(unk_token = unk_token))
        trainer = WordPieceTrainer(special_tokens = spl_tokens)
    else:
        tokenizer = Tokenizer(WordLevel(unk_token = unk_token))
        trainer = WordLevelTrainer(special_tokens = spl_tokens)

    tokenizer.pre_tokenizer = Whitespace()
    return tokenizer, trainer


def train_tokenizer(files, alg='WLV'):
    """
    Takes the files and trains the tokenizer.
    """
    tokenizer, trainer = prepare_tokenizer_trainer(alg)
    tokenizer.train(files, trainer) # training the tokenzier
    tokenizer.save("./tokenizer-trained.json")
    tokenizer = Tokenizer.from_file("./tokenizer-trained.json")
    return tokenizer

def tokenize(input_string, tokenizer):
    """
    Tokenizes the input string using the tokenizer provided.
    """
    output = tokenizer.encode(input_string)
    return output


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
##training on a small dataset
small_file = ['/content/drive/MyDrive/PLN-2024-I/Tokenizadores/Tarea-1/labiblia2.txt']
large_files = ['real_news.csv']
#large_files= large_files.encode('utf-8')
tokens_dict = {}

for files in [small_file, large_files]:
    print(f"========Using vocabulary from {files}=======")
    for alg in ['WLV', 'BPE', 'UNI', 'WPC']:
        trained_tokenizer = train_tokenizer(files, alg)
        input_string = "Todos los bachilleres quieren estudiar en Univalle"
        output = tokenize(input_string, trained_tokenizer)
        tokens_dict[alg] = output.tokens
        print("----", alg, "----")
        print(output.tokens, "->", len(output.tokens))


========Using vocabulary from ['/content/drive/MyDrive/PLN-2024-I/Tokenizadores/Tarea-1/labiblia2.txt']=======
---- WLV ----
['Todos', 'los', '<UNK>', 'quieren', 'estudiar', 'en', '<UNK>'] -> 7
---- BPE ----
['Todos', 'los', 'ba', 'chi', 'll', 'eres', 'quieren', 'estudiar', 'en', 'Uni', 'valle'] -> 11
---- UNI ----
['Todo', 's', 'los', 'ba', 'c', 'h', 'iller', 'es', 'quiere', 'n', 'es', 'tu', 'di', 'ar', 'en', 'Un', 'i', 'valle'] -> 18
---- WPC ----
['Todos', 'los', 'ba', '##chil', '##le', '##res', 'quieren', 'estudiar', 'en', 'Un', '##iva', '##lle'] -> 12
========Using vocabulary from ['real_news.csv']=======
---- WLV ----
['Todos', 'los', '<UNK>', 'quieren', 'estudiar', 'en', '<UNK>'] -> 7
---- BPE ----
['Todos', 'los', 'bach', 'iller', 'es', 'quieren', 'estudiar', 'en', 'Uni', 'valle'] -> 10


# SentencePiece

Es uno de lo tokenizadores de sobpalabras más importantes, a diferencia de los anteriores tokenizadores  que usan pretokenizadores para procesar la entrada, SentencePiece no, el cual trata la entrada como un flujo de entrada sin procesar y todo el procesamiento lo realiza de  manera simultánea haciendo más rápido el tokenizador.  Como un todo,  SentencePiece es un tokenizador de texto no supervisado y destokenizador que resuelve en principio el no tener el texto pretokenizado. SentencePiece implementa la tokenización de subpalabras usando (BPE) [Sennrich et al.]) y el modelo de lenguaje unigrama [Kudo.]) El siguiente es un ejemplo del uso del tokenizador SetencePiece  con el transformer [RoBERTuito](https://huggingface.co/pysentimiento/robertuito-base-cased) que es un modelo de lenguaje preentrenado para contenido generado por el usuario en español, entrenado siguiendo las directrices de RoBERTa en 500 millones de tuits.

In [ ]:
pip install transformers

In [ ]:
pip install pysentimiento

In [ ]:
from transformers import AutoTokenizer

from pysentimiento.preprocessing import preprocess_tweet

tokenizer = AutoTokenizer.from_pretrained('pysentimiento/robertuito-base-cased')

text = "Esto es un tweet estoy usando #Robertuito @pysentimiento 🤣"
preprocessed_text = preprocess_tweet(text)

tokenizer.tokenize(preprocessed_text)
# ['<s>','▁Esto','▁es','▁un','▁tweet','▁estoy','▁usando','▁','▁hashtag','▁','▁ro','bert','uito','▁@usuario','▁','▁emoji','▁cara','▁revolviéndose','▁de','▁la','▁risa','▁emoji','</s>']


In [ ]:
from transformers import AutoTokenizer

from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

text = "Esto es un tweet estoy usando #Robertuito @pysentimiento 🤣"
preprocessed_text = preprocess_tweet(text)

tokenizer.tokenize(preprocessed_text)